# Final Assignment: Tesla & GameStop - Giá cổ phiếu vs Doanh thu

In [ ]:
!pip install yfinance bs4 lxml html5lib nbformat plotly

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from io import StringIO
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data.copy(); revenue_data = revenue_data.copy()
    stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.tz_localize(None)
    revenue_data['Date'] = pd.to_datetime(revenue_data['Date'])
    s = stock_data[stock_data.Date <= '2021-06-14']
    r = revenue_data[revenue_data.Date <= '2021-04-30']
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    fig.add_trace(go.Scatter(x=s.Date, y=s.Close.astype('float'), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=r.Date, y=r.Revenue.astype('float'), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

# Hàm dùng chung để lấy bảng doanh thu theo quý từ trang HTML
def get_revenue(url, match):
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    table = pd.read_html(StringIO(html), match=match)[0]
    table = table.iloc[:, :2]
    table.columns = ["Date", "Revenue"]
    table["Revenue"] = table["Revenue"].astype(str).str.replace(r"[$,]", "", regex=True)
    table = table[(table["Revenue"] != "") & (table["Revenue"] != "nan")]
    return table.dropna().reset_index(drop=True)

## Question 1 - Tesla stock (yfinance)

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 - Tesla revenue (web scraping)

In [ ]:
# Nếu đề bài (lab) cung cấp URL khác, hãy thay vào đây
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = get_revenue(url_tesla, "Tesla Quarterly Revenue")
tesla_revenue.tail()

## Question 3 - GameStop stock (yfinance)

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 - GameStop revenue (web scraping)

In [ ]:
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = get_revenue(url_gme, "GameStop Quarterly Revenue")
gme_revenue.tail()

## Question 5 - Tesla dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6 - GameStop dashboard

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')